# Getting Started with the NeMo Agent Toolkit

In this notebook, we walk through the basics of using the toolkit, from installation all the way to creating and running your very own custom workflow.

## Environment Setup

Ensure you meet the following prerequisites:
1. Python 3.11 or 3.12 installed in a virtual environment.
2. Install `nvidia-nat` as show below:


In [ ]:
%pip install nvidia-nat[langchain, semantic-kernel]

In [1]:
%load_ext autoreload

### Set API keys

In [ ]:
import getpass
import os

if "NVIDIA_API_KEY" not in os.environ:
    nvidia_api_key = getpass.getpass("Enter your NVIDIA API key: ")
    os.environ["NVIDIA_API_KEY"] = nvidia_api_key

if "TAVILY_API_KEY" not in os.environ:
    tavily_api_key = getpass.getpass("Enter your Tavily API key: ")
    os.environ["TAVILY_API_KEY"] = tavily_api_key

if "OPENAI_API_KEY" not in os.environ:
    openai_api_key = getpass.getpass("Enter your OpenAI API key: ")
    os.environ["OPENAI_API_KEY"] = openai_api_key

## Bringing an Agent into the NeMo-Agent-Toolkit

NeMo Agent toolkit works side-by-side and complements any existing agentic framework or memory tool you're using and isn't tied to any specific agentic framework, long-term memory, or data source. This allows you to use your current technology stack - such as LangChain, LlamaIndex, CrewAI, and Microsoft Semantic Kernel, as well as customer enterprise frameworks and simple Python agents - without replatforming.

We'll walk you through how to achieve this.

To demonstrate this, let's say that you have the following simple langchain agent that answers generic user queries about current events by performing a web search using Tavily. We will show you how to bring this agent into the NeMo-Agent-Toolkit and benefit from the configurability, resuability, and easy user experience.

Run the following two cells to create the langchain agent and run it with an example input.

In [30]:
# %load langchain_sample/langchain_agent.py
# SPDX-FileCopyrightText: Copyright (c) 2025, NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
import os

from langchain import hub
from langchain.agents import AgentExecutor, create_react_agent
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_nvidia_ai_endpoints import ChatNVIDIA

# Initialize a tool to search the web
tavily_kwargs = {"max_results": 2, "api_key": os.getenv("TAVILY_API_KEY")}
search = TavilySearchResults(**tavily_kwargs)

# Create a list of tools for the agent
tools = [search]

# Initialize a LLM client
llm_kwargs = {
    "model_name": "meta/llama-3.3-70b-instruct",
    "temperature": 0.0,
    "max_completion_tokens": 1024,
    "api_key": os.getenv("NVIDIA_API_KEY"),
}
llm = ChatNVIDIA(**llm_kwargs)

# Use an open source prompt
prompt = hub.pull("hwchase17/react-chat")

# Initialize a ReAct agent
react_agent = create_react_agent(llm=llm, tools=tools, prompt=prompt, stop_sequence=["\nObservation"])

# Initialize an agent executor to iterate through reasoning steps
agent_executor = AgentExecutor(
    agent=react_agent, tools=tools, max_iterations=15, handle_parsing_errors=True, verbose=True
)

# Invoke the agent with a user query
response = agent_executor.invoke({"input": "Who is the current Pope?", "chat_history": []})

# Print the response
print(response["output"])



> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: tavily_search_results_json
Action Input: "Who is the current Pope?"
Observ[{'title': 'Leo XIV is the new Pope - Vatican News', 'url': 'https://www.vaticannews.va/en/pope/news/2025-05/cardinal-elected-pope-papal-name.html', 'content': 'Just a few moments ago, from the central loggia of Saint Peter\'s Basilica, Cardinal Protodeacon Dominique Mamberti pronounced the formula "Habemus Papam," proclaiming to the city of Rome and to the whole world the news of the election of Robert Francis Cardinal Prevost as Pope Leo XIV\n\nThank you for reading our article. You can keep up-to-date by subscribing to our daily newsletter. Just click here\n\nYour contribution for a great mission:support us in bringing the Pope\'s words into every home [...] ##### More upcoming events:\n\nThe Pope\'s Agenda\n\nListen to our podcasts\nListen to our podcasts\nSubscribe to our newsletters\nSubscribe to our newsletters\nAngelus\n

## Moving an Agent into the NeMo-Agent-Toolkit

In this notebook, we walk through the process of moving an existing agent into the NeMo-Agent-Toolkit. We will use the `first_search_agent` as an example which is identical to the pure LangChain agent we created in the previous step. Only differences here are the addition of the configuration object and wrapping the agent in a function with a decorator.



In [31]:
import logging

from pydantic import Field

from nat.builder.builder import Builder
from nat.builder.framework_enum import LLMFrameworkEnum
from nat.builder.function_info import FunctionInfo
from nat.cli.register_workflow import register_function
from nat.data_models.function import FunctionBaseConfig

logger = logging.getLogger(__name__)


class FirstSearchAgentFunctionConfig(FunctionBaseConfig, name="first_search_agent"):
    """
    Configuration object for the first search agent.
    """

    pass


@register_function(config_type=FirstSearchAgentFunctionConfig, framework_wrappers=[LLMFrameworkEnum.LANGCHAIN])
async def first_search_agent_function(_config: FirstSearchAgentFunctionConfig, _builder: Builder):
    import os

    from langchain import hub
    from langchain.agents import AgentExecutor, create_react_agent
    from langchain_community.tools.tavily_search import TavilySearchResults
    from langchain_nvidia_ai_endpoints import ChatNVIDIA

    # Initialize a tool to search the web
    tavily_kwargs = {"max_results": 2, "api_key": os.getenv("TAVILY_API_KEY")}
    search = TavilySearchResults(**tavily_kwargs)

    # Create a list of tools for the agent
    tools = [search]

    # Initialize a LLM client
    llm_kwargs = {
        "model_name": "meta/llama-3.3-70b-instruct",
        "temperature": 0.0,
        "max_tokens": 1024,
        "api_key": os.getenv("NVIDIA_API_KEY"),
    }
    llm = ChatNVIDIA(**llm_kwargs)

    # Use an open source prompt
    prompt = hub.pull("hwchase17/react-chat")

    # Initialize a ReAct agent
    react_agent = create_react_agent(llm=llm, tools=tools, prompt=prompt, stop_sequence=["\nObservation"])

    # Initialize an agent executor to iterate through reasoning steps
    agent_executor = AgentExecutor(
        agent=react_agent, tools=tools, max_iterations=15, handle_parsing_errors=True, verbose=True
    )

    # Setup a function which will execute the agent. This will be executed each time the agent is invoked.
    async def _response_fn(input_message: str) -> str:
        response = agent_executor.invoke({"input": input_message, "chat_history": []})

        return response["output"]

    yield FunctionInfo.from_fn(_response_fn)

A function with the same config type `__main__/first_search_agent` has already been registered. Overriding the previous registration since this is an interactive mode. This is not recommended and may cause unexpected behavior. Please ensure that the function is only registered once.


## Running the Agent in the NeMo-Agent-Toolkit

Now that we have the agent configured and registered, we can instantiate it and run it using the NeMo-Agent-Toolkit.

In [34]:
from nat.data_models.config import Config
from nat.runtime.loader import load_workflow

# Create a configuration object
config = Config(
    workflow=FirstSearchAgentFunctionConfig(),
)

# Load the workflow
async with load_workflow(config) as workflow:
    # Run the agent with the input.
    async with workflow.run("Who is the current Pope?") as runner:
        print(await runner.result(to_type=str))

    # Run multiple times with different inputs.
    async with workflow.run("When were wolves reintroduced to Colorado?") as runner:
        print(await runner.result(to_type=str))


Failed to import plugin 'aiq_inference_time_scaling'
Traceback (most recent call last):
  File "/home/mdemoret/Repos/AgentIQ/NeMo-Agent-Toolkit-dev2/src/nat/runtime/loader.py", line 187, in discover_and_register_plugins
    entry_point.load()
  File "/home/mdemoret/.local/share/uv/python/cpython-3.11.13-linux-x86_64-gnu/lib/python3.11/importlib/metadata/__init__.py", line 202, in load
    module = import_module(match.group('module'))
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/mdemoret/.local/share/uv/python/cpython-3.11.13-linux-x86_64-gnu/lib/python3.11/importlib/__init__.py", line 126, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1204, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1176, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1126, in _find_and_load_unlocked
  File "<frozen importlib.



> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: tavily_search_results_json
Action Input: "Who is the current Pope?"
Observ[{'title': 'Leo XIV is the new Pope - Vatican News', 'url': 'https://www.vaticannews.va/en/pope/news/2025-05/cardinal-elected-pope-papal-name.html', 'content': 'Just a few moments ago, from the central loggia of Saint Peter\'s Basilica, Cardinal Protodeacon Dominique Mamberti pronounced the formula "Habemus Papam," proclaiming to the city of Rome and to the whole world the news of the election of Robert Francis Cardinal Prevost as Pope Leo XIV\n\nThank you for reading our article. You can keep up-to-date by subscribing to our daily newsletter. Just click here\n\nYour contribution for a great mission:support us in bringing the Pope\'s words into every home [...] ##### More upcoming events:\n\nThe Pope\'s Agenda\n\nListen to our podcasts\nListen to our podcasts\nSubscribe to our newsletters\nSubscribe to our newsletters\nAngelus\n

## Adding Configuration Settings to the Agent

When creating an agent, you may want to add additional configuration settings to the agent. This can be done by adding properties to the configuration object. Configuration settings can even include LLM settings, tools, and even other agents.

To make LLM settings configurable, you can add the following properties to the configuration object:
```python
class FirstSearchAgentFunctionConfig(FunctionBaseConfig, name="first_search_agent"):
    llm_name: LLMRef = Field(description="The name of the LLM to use")
```

And then in your function, you would replace the following line:
```python
# Initialize a LLM client
llm_kwargs = {
    "model_name": "meta/llama-3.3-70b-instruct",
    "temperature": 0.0,
    "max_tokens": 1024,
    "api_key": os.getenv("NVIDIA_API_KEY"),
}
llm = ChatNVIDIA(**llm_kwargs)
```
with
```python
# Initialize a LLM client
llm = await builder.get_llm(config.llm_name, wrapper_type=LLMFrameworkEnum.LANGCHAIN)
```

## Updated Agent in NeMo Agent Toolkit

In [ ]:
import logging

from pydantic import Field

from nat.builder.builder import Builder
from nat.builder.framework_enum import LLMFrameworkEnum
from nat.builder.function_info import FunctionInfo
from nat.cli.register_workflow import register_function
from nat.data_models.component_ref import LLMRef
from nat.data_models.function import FunctionBaseConfig

logger = logging.getLogger(__name__)


class FirstSearchAgentFunctionConfig(FunctionBaseConfig, name="first_search_agent"):
    """
    NeMo Agent toolkit function template. Please update the description.
    """

    llm_name: LLMRef = Field(description="The name of the LLM to use")
    verbose: bool = Field(default=True, description="Whether to print verbose output")


@register_function(config_type=FirstSearchAgentFunctionConfig, framework_wrappers=[LLMFrameworkEnum.LANGCHAIN])
async def first_search_agent_function(config: FirstSearchAgentFunctionConfig, builder: Builder):
    import os

    from langchain import hub
    from langchain.agents import AgentExecutor, create_react_agent
    from langchain_community.tools.tavily_search import TavilySearchResults
    from langchain_nvidia_ai_endpoints import ChatNVIDIA

    # Initialize a tool to search the web
    tavily_kwargs = {"max_results": 2, "api_key": os.getenv("TAVILY_API_KEY")}
    search = TavilySearchResults(**tavily_kwargs)

    # Create a list of tools for the agent
    tools = [search]

    # Get the LLM client from the builder
    llm = await builder.get_llm(config.llm_name, wrapper_type=LLMFrameworkEnum.SEMANTIC_KERNEL)

    # Use an open source prompt
    prompt = hub.pull("hwchase17/react-chat")

    # Initialize a ReAct agent
    react_agent = create_react_agent(llm=llm, tools=tools, prompt=prompt, stop_sequence=["\nObservation"])

    # Initialize an agent executor to iterate through reasoning steps
    agent_executor = AgentExecutor(
        agent=react_agent, tools=tools, max_iterations=15, handle_parsing_errors=True, verbose=config.verbose
    )

    async def _response_fn(input_message: str) -> str:
        response = agent_executor.invoke({"input": input_message, "chat_history": []})

        return response["output"]

    yield FunctionInfo.from_fn(_response_fn)

A function with the same config type `__main__/first_search_agent` has already been registered. Overriding the previous registration since this is an interactive mode. This is not recommended and may cause unexpected behavior. Please ensure that the function is only registered once.


In [36]:
from nat.data_models.config import Config
from nat.llm.nim_llm import NIMModelConfig
from nat.runtime.loader import load_workflow

config = Config(
    llms={"nim_llm": NIMModelConfig(model_name="meta/llama-3.3-70b-instruct", temperature=0.0, max_tokens=1024)},
    workflow=FirstSearchAgentFunctionConfig(llm_name="nim_llm", verbose=True),
)

async with load_workflow(config) as workflow:
    async with workflow.run("Who is the current Pope?") as runner:
        print(await runner.result(to_type=str))

Failed to import plugin 'aiq_inference_time_scaling'
Traceback (most recent call last):
  File "/home/mdemoret/Repos/AgentIQ/NeMo-Agent-Toolkit-dev2/src/nat/runtime/loader.py", line 187, in discover_and_register_plugins
    entry_point.load()
  File "/home/mdemoret/.local/share/uv/python/cpython-3.11.13-linux-x86_64-gnu/lib/python3.11/importlib/metadata/__init__.py", line 202, in load
    module = import_module(match.group('module'))
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/mdemoret/.local/share/uv/python/cpython-3.11.13-linux-x86_64-gnu/lib/python3.11/importlib/__init__.py", line 126, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1204, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1176, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1126, in _find_and_load_unlocked
  File "<frozen importlib.



> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: tavily_search_results_json
Action Input: current Pope[{'title': 'May 8, 2025 Leo XIV elected as first American pope', 'url': 'https://www.cnn.com/world/live-news/new-pope-conclave-day-two-05-08-25', 'content': '• About the new pope: Leo, a 69-year-old from Chicago, is a leader with global experience. He spent much of his career as a missionary in South America and holds dual citizenship in the US and Peru, where he served as a bishop. He most recently led a powerful Vatican office for bishop appointments. He is expected to build on Pope Francis’ reforms. [...] Young people gather on the steops of the Basilica of the National Shrine of the Immaculate Conception in Washington, DC, on Thursday after the announcement of the new pope.\n\nFrom young seminarians in robes to college students in shorts and T-shirts, the gathering in front of the Basilica of the National Shrine of the Immaculate Conception in W

#### Creating a new NeMo-Agent-Toolkit Workflow 

Bringing this agent into the toolkit requires creating a new workflow and configuring the tools, and so on. A workflow is a self-contained pipeline that orchestrates tools (e.g., custom arithmetic tools, web search, RAG) and one or more LLMs to process user inputs and generate outputs.

With our `nat workflow create` sub-command, you can scaffold and register new workflows within seconds. 

For example, to create an agent called `first_search_agent` in `.tmp/notebooks` you would run the following commands. 

> Note: The agent in this example has already been created in `examples/notebooks/first_search_agent` directory.

```bash
mkdir -p $PROJECT_ROOT/.tmp/notebooks
nat workflow create --workflow-dir $PROJECT_ROOT/.tmp/notebooks/first_search_agent
```

Expected Cell Output:
```bash
Installing workflow 'first_search_agent'...
Workflow 'first_search_agent' installed successfully.
Workflow 'first_search_agent' created successfully in '/NeMo-Agent-Toolkit/.tmp/notebooks/first_search_agent'.
```


The above command:
- Creates a new directory similar to `examples/notebooks/first_search_agent`.
- Sets up the necessary files and folders.
- Installs the new Python package for your workflow.

The registration process is built around two main components:
1. **A configuration class that inherits from `WorkflowBaseConfig`**
    
    Configuration classes that inherit from `TypedBaseModel` and `BaseModelRegistryTag` serve as Pydantic-based configuration objects that define both the plugin type identifier and runtime configuration settings for each NeMo Agent toolkit component. Each plugin type (functions, LLMs, embedders, retrievers, memory, front-ends, etc.) has its own base configuration class (e.g., `FunctionBaseConfig`, `LLMBaseConfig`, `EmbedderBaseConfig`) that establishes the plugin category, while concrete implementations specify a unique name parameter that automatically populates the type field for plugin identification. These configuration classes encapsulate runtime parameters as typed Pydantic fields with validation rules, default values, and documentation (e.g., `api_key`, `model_name`, `temperature` for LLM providers, or `uri`, `collection_name`, `top_k` for retrievers), enabling type-safe configuration management, automatic schema generation, and validation across the entire plugin ecosystem.

2. **A decorated async function (with `@register_workflow`) that yields a callable response function.**
     
     A `FunctionInfo` object is a structured representation yielded from functions decorated with `@register_function` that serves as a framework-agnostic wrapper for callable functions in the NeMo Agent Toolkit. This object encapsulates the function's main callable (e.g., `_response_fn`) that will be invoked at runtime, along with its input/output Pydantic schemas for validation, description for documentation, and optional type converters for automatic type transformation. FunctionInfo objects provide a consistent interface that can be dynamically translated into framework-specific representations (e.g., LangChain tools, LlamaIndex functions) at runtime or invoked directly as standard Python async coroutines, enabling seamless integration across different LLM frameworks while maintaining type safety and validation.


Once configured, you can run workflows via the command line (`nat run`) or launch them as services (`nat serve`) to handle requests in real time.

#### Customizing your Workflow

Now its time to define the same langchain agent inside your newly created workflow. This is as simple as making a few code additions to the `first_search_agent_function`.
- Add langchain framework wrappers (all this does is indicate which framework you are wrapping your code in which enables profiling the workflow later)
- Paste your agent initialization code inside the `first_search_agent_function`
- Paste your agent invocation code inside the `_response_fn` function

Your final `first_search_agent_function.py` should look like:

In [ ]:
import logging

from pydantic import Field

from nat.builder.builder import Builder
from nat.builder.framework_enum import LLMFrameworkEnum
from nat.builder.function_info import FunctionInfo
from nat.cli.register_workflow import register_function
from nat.data_models.component_ref import LLMRef
from nat.data_models.function import FunctionBaseConfig

logger = logging.getLogger(__name__)


class FirstSearchAgentFunctionConfig(FunctionBaseConfig, name="first_search_agent"):
    """
    NeMo Agent toolkit function template. Please update the description.
    """

    llm_name: LLMRef = Field(description="The name of the LLM to use")
    verbose: bool = Field(default=True, description="Whether to print verbose output")


@register_function(config_type=FirstSearchAgentFunctionConfig, framework_wrappers=[LLMFrameworkEnum.LANGCHAIN])
async def first_search_agent_function(config: FirstSearchAgentFunctionConfig, builder: Builder):
    import os

    from langchain import hub
    from langchain.agents import AgentExecutor, create_react_agent
    from langchain_community.tools.tavily_search import TavilySearchResults
    from langchain_nvidia_ai_endpoints import ChatNVIDIA

    # Initialize a tool to search the web
    tavily_kwargs = {"max_results": 2, "api_key": os.getenv("TAVILY_API_KEY")}
    search = TavilySearchResults(**tavily_kwargs)

    # Create a list of tools for the agent
    tools = [search]

    # Get the LLM client from the builder
    llm = await builder.get_llm(config.llm_name, wrapper_type=LLMFrameworkEnum.LANGCHAIN)

    # Use an open source prompt
    prompt = hub.pull("hwchase17/react-chat")

    # Initialize a ReAct agent
    react_agent = create_react_agent(llm=llm, tools=tools, prompt=prompt, stop_sequence=["\nObservation"])

    # Initialize an agent executor to iterate through reasoning steps
    agent_executor = AgentExecutor(
        agent=react_agent, tools=tools, max_iterations=15, handle_parsing_errors=True, verbose=config.verbose
    )

    async def _response_fn(input_message: str) -> str:
        response = agent_executor.invoke({"input": input_message, "chat_history": []})

        return response["output"]

    yield FunctionInfo.from_fn(_response_fn)


Once you have your workflow registered, you can reference it by its `_type` in a YAML file. 

For example:

```yaml
workflow:
  _type: first_search_agent

  # Settings for the agent
  llm_name: nim_llm
  verbose: true
```


#### Running your Workflow

The NeMo Agent toolkit provides several ways to run/host an workflow. These are called `front_end` plugins. Some examples are:

console: `nat run` (or long version nat start console …). This is useful when performing local testing and debugging. It allows you to pass inputs defined as arguments directly into the workflow. This is show already in the notebook.

Fastapi: `nat serve`(or long version nat start fastapi …). This is useful when hosting your workflow as a REST and websockets endpoint.

MCP: `nat mcp` (or long version nat start mcp …). This is useful when hosting the workflow and/or any function as an MCP server

While these are the built in front-end components, the system is extensible with new user defined front-end plugins.

For more info, here is a good resource for using the various plugins from the CLI: [cli.md](https://github.com/NVIDIA/NeMo-Agent-Toolkit/blob/develop/docs/source/reference/cli.md)

In order to test your new agent using the console, run:

In [37]:
!nat run --config_file first_search_agent/configs/config.yml --input "Who is the current Pope?" --input "What is the most common type of aardvark?"

71836.69s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


2025-09-04 17:21:25,338 - nat.cli.commands.start - INFO - Starting NAT from config file: 'first_search_agent/configs/config.yml'
/home/mdemoret/Repos/AgentIQ/NeMo-Agent-Toolkit-dev2/examples/notebooks/first_search_agent/src/nat_first_search_agent/first_search_agent_function.py:48: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-tavily package and should be used instead. To use it run `pip install -U :class:`~langchain-tavily` and import as `from :class:`~langchain_tavily import TavilySearch``.
  search = TavilySearchResults(**tavily_kwargs)

Configuration Summary:
--------------------
Workflow Type: first_search_agent
Number of Functions: 0
Number of LLMs: 0
Number of Embedders: 0
Number of Memory: 0
Number of Object Stores: 0
Number of Retrievers: 0
Number of TTC Strategies: 0
Number of Authentication Providers: 0



> Entering new AgentExecutor 

As shown above, this will return the same output as your previously created langchain agent.

#### Runtime Configurations

To benefit from the configurability of this toolkit, we can update the configuration object and config file along with the function to use the parameters at runtime.

This involves allowing the toolkit to sets up your tools, LLM, and any additional logic like maximum number of historical messages to provide to the agent, maximum number of iterations to run the agent, description of the agent and so on.

The toolkit will make use of the `Builder` class to utilize them at runtime.

Your final configuration object should look like this:
```python
class SecondSearchAgentFunctionConfig(FunctionBaseConfig, name="second_search_agent"):
    """
    NeMo Agent toolkit function template. Please update the description.
    """
    tool_names: list[FunctionRef] = Field(default=[], description="List of tool names to use")
    llm_name: LLMRef = Field(description="LLM name to use")
    max_history: int = Field(default=10, description="Maximum number of historical messages to provide to the agent")
    max_iterations: int = Field(default=15, description="Maximum number of iterations to run the agent")
    handle_parsing_errors: bool = Field(default=True, description="Whether to handle parsing errors")
    verbose: bool = Field(default=True, description="Whether to print verbose output")
    description: str = Field(default="", description="Description of the agent")
```

You can then replace:
```python
tool = [search]
```
with 
```python
tools = builder.get_tools(config.tool_names, wrapper_type=LLMFrameworkEnum.LANGCHAIN)
```
> **Note**: This allows you to bring in tools from other frameworks like llama index as well and wrap them with langchain since you are implementing your agent in langchain.

In a similar way, you can initialize your llm by utilizing the parameters from the configuration object in the following way:
```python
llm = await builder.get_llm(config.llm_name, wrapper_type=LLMFrameworkEnum.LANGCHAIN)
```

For each tool or reusable plugin, there are potentially multiple optional parameters with default values that can be overridden. The `nat info components` command can be used to list all available parameters. For example, to list all available parameters for the LLM nim type run:

```bash
nat info components -t llm_provider -q nim
```

#### Reusing the Inbuilt Tavily Search Function

We can also make use of some of many example functions that the toolkit provides for common use cases. In this agent example, rather than reimplementing the tavily search, we will use the inbuilt function for internet search which is built on top of langchain's tavily search API. You can list available functions using the following:

In [39]:
!nat info components -t function

71914.19s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


                               NAT Search Results                               
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ package       ┃ version       ┃ component_ty… ┃ component_n… ┃ description   ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ nat_profiler_ │ 1.3.0a2.dev33 │ function      │ flow_chart   │ Configuration │
│ agent         │ +g8a08f33a    │               │              │ for the       │
│               │               │               │              │ FlowChart     │
│               │               │               │              │ tool.         │
│               │               │               │              │               │
│               │               │               │              │   Args:       │
│               │               │               │              │     _type     │
│               │               │               │              │ (str): The    │
│               │           

This function can be used any number of times in the configuration YAML by specifying the `_type` as `tavily_internet_search`

```yaml
functions:
  my_internet_search:
    _type: tavily_internet_search
    max_results: 2
    api_key: $TAVILY_API_KEY
```

#### Final Code and Configuration
The final code for your workflow can be found in [this example](examples/my_agent_workflow/src/nat_my_agent_workflow/my_agent_workflow_function.py)

In [ ]:
# %load first_search_agent/src/nat_first_search_agent/second_search_agent_function.py
# SPDX-FileCopyrightText: Copyright (c) 2025, NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import logging

from pydantic import Field

from nat.builder.builder import Builder
from nat.builder.framework_enum import LLMFrameworkEnum
from nat.builder.function_info import FunctionInfo
from nat.cli.register_workflow import register_function
from nat.data_models.component_ref import FunctionRef, LLMRef
from nat.data_models.function import FunctionBaseConfig

logger = logging.getLogger(__name__)


class SecondSearchAgentFunctionConfig(FunctionBaseConfig, name="second_search_agent"):
    """
    NeMo Agent toolkit function template. Please update the description.
    """

    tool_names: list[FunctionRef] = Field(default=[], description="List of tool names to use")
    llm_name: LLMRef = Field(description="LLM name to use")
    max_history: int = Field(default=10, description="Maximum number of historical messages to provide to the agent")
    max_iterations: int = Field(default=15, description="Maximum number of iterations to run the agent")
    handle_parsing_errors: bool = Field(default=True, description="Whether to handle parsing errors")
    verbose: bool = Field(default=True, description="Whether to print verbose output")
    description: str = Field(default="", description="Description of the agent")


@register_function(config_type=SecondSearchAgentFunctionConfig, framework_wrappers=[LLMFrameworkEnum.LANGCHAIN])
async def second_search_agent_function(config: SecondSearchAgentFunctionConfig, builder: Builder):
    from langchain import hub
    from langchain.agents import AgentExecutor, create_react_agent

    # Create a list of tools for the agent
    tools = builder.get_tools(config.tool_names, wrapper_type=LLMFrameworkEnum.LANGCHAIN)

    llm = await builder.get_llm(config.llm_name, wrapper_type=LLMFrameworkEnum.LANGCHAIN)

    # Use an open source prompt
    prompt = hub.pull("hwchase17/react-chat")

    # Initialize a ReAct agent
    react_agent = create_react_agent(llm=llm, tools=tools, prompt=prompt, stop_sequence=["\nObservation"])

    # Initialize an agent executor to iterate through reasoning steps
    agent_executor = AgentExecutor(
        agent=react_agent,
        tools=tools,
        max_iterations=config.max_iterations,
        handle_parsing_errors=config.handle_parsing_errors,
        verbose=config.verbose,
    )

    async def _response_fn(input_message: str) -> str:
        response = await agent_executor.ainvoke({"input": input_message, "chat_history": []})

        return response["output"]

    yield FunctionInfo.create(single_fn=_response_fn)


The final configuration file should resemble the following:

In [5]:
# %load first_search_agent/configs/config_modified.yml
# SPDX-FileCopyrightText: Copyright (c) 2025, NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
general:
  use_uvloop: true
  logging:
    console:
      _type: console
      level: WARN

llms:
  nim_llm:
    _type: nim
    model_name: meta/llama-3.3-70b-instruct
    temperature: 0.0
    max_tokens: 1024
    api_key: $NVIDIA_API_KEY

functions:
  my_internet_search:
    _type: tavily_internet_search
    max_results: 2
    api_key: $TAVILY_API_KEY

workflow:
  _type: second_search_agent
  tool_names:
    - my_internet_search
  llm_name: nim_llm
  max_history: 10
  max_iterations: 15
  description: "A helpful assistant that can search the internet for information"


In [40]:
!nat run --config_file first_search_agent/configs/config_modified.yml --input "Who is the current Pope?"

72007.49s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


2025-09-04 17:24:16,049 - nat.cli.commands.start - INFO - Starting NAT from config file: 'first_search_agent/configs/config_modified.yml'

Configuration Summary:
--------------------
Workflow Type: second_search_agent
Number of Functions: 1
Number of LLMs: 1
Number of Embedders: 0
Number of Memory: 0
Number of Object Stores: 0
Number of Retrievers: 0
Number of TTC Strategies: 0
Number of Authentication Providers: 0



> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: my_internet_search
Action Input: Who is the current Pope?/home/mdemoret/Repos/AgentIQ/NeMo-Agent-Toolkit-dev2/packages/nvidia_nat_langchain/src/nat/plugins/langchain/tools/tavily_internet_search.py:45: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-tavily package and should be used instead. To use it run `pip install -U :class:`~langchain-tavily` and

#### NAT Serve

You can also use the `nat serve` sub-command to launch a server and make HTTP requests to the endpoints as shown below. Refer to [this documentation](https://docs.nvidia.com/nemo/agent-toolkit/latest/reference/api-server-endpoints.html) for more information on available endpoints.

In [41]:
%%bash --bg
# This will start background nat service and might take a moment to be ready
nat serve --config_file first_search_agent/configs/config_modified.yml

In [42]:
%%bash
# Issue a request to the background service
curl --request POST \
  --url http://localhost:8000/chat \
  --header 'Content-Type: application/json' \
  --data '{
    "messages": [
      {
        "role": "user",
        "content": "Who is the current Pope?"
      }
    ]
}'
# Terminate the process after completion
pkill -9 nat

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0

100   513  100   400  100   113      6      1  0:01:53  0:00:59  0:00:54   103:00:58  0:00:55     00   113      0      1  0:01:53  0:00:59  0:00:54     0


{"id":"0f199570-e339-4596-a84b-e92b277822ce","object":"chat.completion","model":"","created":1757021244,"choices":[{"message":{"content":"The current Pope is Pope Leo XIV, formerly known as Robert Francis Cardinal Prevost.","role":null},"delta":null,"finish_reason":"stop","index":0}],"usage":{"prompt_tokens":0,"completion_tokens":14,"total_tokens":14},"system_fingerprint":null,"service_tier":null}

pkill: killing pid 2051 failed: Operation not permitted


#### Reusing the Inbuilt ReAct Agent

NeMo Agent Toolkit has a reusable react agent function. We can reuse that agent here to simplify the workflow even further.

In [ ]:
!nat info components -t function -q react_agent

In [ ]:
%load first_search_agent/configs/config_react_agent.yml

In [ ]:
!nat run --config_file first_search_agent/configs/config_react_agent.yml --input "Who is the current Pope?"